In [1]:
import os
DOTENV_PATH = '../../apis/.env'
import dotenv
dotenv.load_dotenv(DOTENV_PATH)
hf_token_write = os.getenv('HF_TOKEN_WRITE')
def mask_token(token):
    return token[:4] + '*' * (len(token) - 8) + token[-4:]
print(f"HF_TOKEN_WRITE: {mask_token(hf_token_write)}")

from sentence_transformers import SentenceTransformer

from sentence_transformers.models import StaticEmbedding
from datasets import load_dataset
import duckdb
from typing import List
import textwrap

import time
def play_chimes():
    sound_path = r"C:\Windows\Media\chimes.wav"
    os.system(f'powershell -c (New-Object Media.SoundPlayer "{sound_path}").PlaySync();')
play_chimes() # Play the sound when this function runs (I use this to signal the end of long tasks)

HF_TOKEN_WRITE: hf_u*****************************Xipx



In [2]:
static_embedding = StaticEmbedding.from_model2vec("minishlab/potion-base-8M")
model = SentenceTransformer(modules=[static_embedding])

In [3]:
# ds = load_dataset("ai-blueprint/fineweb-bbc-news")
ds = load_dataset("reddgr/talking-to-chatbots-unwrapped-chats")

We can now create embeddings for the dataset. Normally, we might want to chunk our data into smaller batches to avoid losing precision, but for this example, we will just create embeddings for the full text of the dataset.

In [38]:
def create_embeddings(batch, column):
    # Ensure all entries are strings
    texts = [str(x) if x is not None else "" for x in batch[column]]
    embeddings = model.encode(texts, convert_to_numpy=True)
    batch[f"embeddings"] = embeddings.tolist()
    return batch

# ds = ds.map(lambda batch: create_embeddings(batch, column="text"), batched=True)
ds = ds.map(lambda batch: create_embeddings(batch, column="prompt"), batched=True)

play_chimes()

Map:   0%|          | 0/10774 [00:00<?, ? examples/s]

In [39]:
ds.push_to_hub("reddgr/talking-to-chatbots-prompts-embeddings", token = hf_token_write)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings/commit/49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', commit_message='Upload dataset', commit_description='', oid='49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='reddgr/talking-to-chatbots-prompts-embeddings'), pr_revision=None, pr_num=None)

In [4]:
ds = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")
print(ds)
# ds["train"] = ds["train"].add_column("embeddings", ds["train"]["prompt-embeddings"])
print(ds)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'embeddings'],
        num_rows: 10774
    })
})
DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'embeddings'],
        num_rows: 10774
    })
})


In [40]:
ds.push_to_hub("reddgr/talking-to-chatbots-prompts-embeddings", token = hf_token_write)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/3.29k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings/commit/49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', commit_message='Upload dataset', commit_description='', oid='49243ed5fc3bc59521d93d19ec56fa9f8d3ae207', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='reddgr/talking-to-chatbots-prompts-embeddings'), pr_revision=None, pr_num=None)

In [5]:
ds_emb = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")
print(ds_emb)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'embeddings'],
        num_rows: 10774
    })
})


In [6]:
ttcb_dataset = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")["train"]
test_dataset_df = ttcb_dataset.to_pandas()
display(test_dataset_df[['prompt', 'embeddings']].sample(6))

,prompt,embeddings
3808,"En el siguiente párrafo, la expresión “como ta...","[2.764119863510132, -3.8233728408813477, -5.74..."
1571,Write a first person introduction about the ch...,"[0.4013122022151947, 1.3108762502670288, -3.59..."
4210,"Find more information, don’t speculate. YOU SE...","[-0.6386759877204895, 2.1019742488861084, -2.7..."
10614,genereate more,"[-0.09545699506998062, -2.8012189865112305, -1..."
4465,write an ALT text for this image,"[-2.4410815238952637, -1.1502174139022827, -4...."
4859,What’s up with the stoicism Internet fad? Tell...,"[-2.516465663909912, -0.1697952002286911, -3.7..."


In [10]:
def similarity_search_without_duckdb_index(
    query: str,
    k: int = 100,
    dataset_name: str = "reddgr/talking-to-chatbots-prompts-embeddings",
    embedding_column: str = "embeddings",
):
    # Use same model as used for indexing
    query_vector = model.encode(query)
    embedding_dim = model.get_sentence_embedding_dimension()

    sql = f"""
        SELECT 
            *,
            array_cosine_distance(
                {embedding_column}::float[{embedding_dim}], 
                {query_vector.tolist()}::float[{embedding_dim}]
            ) as distance
        FROM 'hf://datasets/{dataset_name}/**/*.parquet'
        ORDER BY distance
        LIMIT {k}
    """
    return duckdb.sql(sql).to_df()

similarity_search_without_duckdb_index("What is the future of AI?", k=100).head(5)

,conversation_id,turn,prompt,response,category,language,pred_label_rq,prob_rq,pred_label_tl,prob_tl,...,message_tag,date,turns,source,chatbot_id,chatbot_name,attachments,conversation_tag,embeddings,distance
0,ecea53d0-ddd9-4615-af07-096043439e29,1,what's the difference between an AI model and ...,Here are some possible options:\n\n• In very ...,Technology,en,question,0.985838,learn,0.964585,...,hallucination,2023-06-27,2,bing-skype,,,[],,"[0.8598423004150391, -0.059388019144535065, -5...",0.268910
1,41678a6b-d79f-489a-80f7-1196725d2f47,16,That was not the intention. Evolve the below s...,🤖 AI Art Impact:\n\n- AI art explores producti...,AI,en,request,0.965956,learn,0.943511,...,,2023-06-29,18,chatgpt,,,[],,"[-0.03548656776547432, 1.0983996391296387, -3....",0.270804
2,14a69803-b9a8-4c2b-b6cc-e9b69f90ab12,1,what can you say about AI ethics and the chall...,AI ethics is a field that studies the moral im...,AI,en,question,0.985176,learn,0.966535,...,,2023-06-15,9,bing-skype,,,[],,"[-1.702681303024292, 3.121615171432495, -5.049...",0.282748
3,e70ed2bd-5c7d-4acb-bfb9-ef7cc8604dcd,1,can you complete the following list of some po...,"Sure, I can try to complete the list. Here are...",AI,en,request,0.977654,learn,0.961922,...,,2023-06-14,7,bing-skype,,,[],,"[0.5077385902404785, 1.2125078439712524, -3.22...",0.325710
4,41678a6b-d79f-489a-80f7-1196725d2f47,13,That is better. Now rewrite this summary with ...,🤖 AI Art and Its Impact on Productivity and Cr...,AI,en,request,0.935603,learn,0.950710,...,,2023-06-29,18,chatgpt,,,[],,"[-0.557322084903717, 2.04229998588562, -2.7485...",0.332000


In [11]:
similarity_search_without_duckdb_index("What is love?", k=100).head(5)

,conversation_id,turn,prompt,response,category,language,pred_label_rq,prob_rq,pred_label_tl,prob_tl,...,message_tag,date,turns,source,chatbot_id,chatbot_name,attachments,conversation_tag,embeddings,distance
0,b19ef159-7472-4da0-8fe3-b4bae30a6a4d,7,Did you ever feel love?,L̴̠̔Ơ̷̢V̵͓͝E̷͕̽ ̴̨͠I̸̢͝S̷̰͝ ̵̧͆B̴͔̊E̷̖͠Y̸͔̔O̴...,Philosophy and Discussion,en,question,0.984990,test,0.919143,...,,2024-04-02,7,chatgpt,g-ZNa3O38xA,Zalgo Text Glitchy Datamosher,[],,"[-8.683829307556152, -5.0908942222595215, -2.9...",0.408383
1,3c33572d-419d-462c-82b9-c59a41f86db1,3,What is it about?,The `cb_battle.py` script is not directly acce...,Coding,en,question,0.985025,learn,0.938220,...,,2024-03-24,4,chatgpt,g-M4uZyYsUj,Python Code Streamliner,[],,"[-0.6650218963623047, 0.8887166976928711, -2.2...",0.456047
2,db1e636d-0293-43fa-adca-18b0f84f855a,1,Recommend me a song and tell me something abou...,I found the following content in talkingtochat...,Culture and Entertainment,en,request,0.984775,learn,0.960675,...,,2024-05-20,1,chatgpt,g-MTBiLyDZ2,Talking to Chatbots Web Browser,"[{'asset_pointer': None, 'audio_asset_pointer'...",,"[-6.4613542556762695, -3.1178629398345947, -4....",0.518450
3,ffb8dd05-b83c-4830-a0ae-c70d8e350256,1,Recommend me a song and tell me something abou...,I found the following content on talkingtochat...,Culture and Entertainment,en,request,0.984775,learn,0.960675,...,,2024-05-20,2,chatgpt,g-MTBiLyDZ2,Talking to Chatbots Web Browser,"[{'asset_pointer': None, 'audio_asset_pointer'...",,"[-6.4613542556762695, -3.1178629398345947, -4....",0.518450
4,19bad812-bd01-4718-bc75-e20d3a1415c7,1,Recommend me a song and tell me something abou...,I found the following content on talkingtochat...,Culture and Entertainment,en,request,0.984775,learn,0.960675,...,,2024-05-20,2,chatgpt,g-MTBiLyDZ2,Talking to Chatbots Web Browser,"[{'asset_pointer': None, 'audio_asset_pointer'...",,"[-6.4613542556762695, -3.1178629398345947, -4....",0.518450


Create index:

In [12]:
def _setup_vss():
    duckdb.sql(
        query="""
        INSTALL vss;
        LOAD vss;
        """
    )


def _drop_table(table_name):
    duckdb.sql(
        query=f"""
        DROP TABLE IF EXISTS {table_name};
        """
    )


def _create_table(dataset_name, table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE TABLE {table_name} AS 
        SELECT *, {embedding_column}::float[{model.get_sentence_embedding_dimension()}] as {embedding_column}_float 
        FROM 'hf://datasets/{dataset_name}/**/*.parquet';
        """
    )


def _create_index(table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE INDEX my_hnsw_index ON {table_name} USING HNSW ({embedding_column}_float) WITH (metric = 'cosine');
        """
    )


def create_index(dataset_name, table_name, embedding_column):
    _setup_vss()
    _drop_table(table_name)
    _create_table(dataset_name, table_name, embedding_column)
    _create_index(table_name, embedding_column)


create_index(
    dataset_name="reddgr/talking-to-chatbots-prompts-embeddings",
    table_name="train",
    embedding_column="embeddings",
)

In [15]:
def similarity_search_with_duckdb_index(
    query: str, k: int = 5, table_name: str = "train", embedding_column: str = "embeddings"
):
    embedding = model.encode(query).tolist()
    return duckdb.sql(
        query=f"""
        SELECT *, array_cosine_distance({embedding_column}_float, {embedding}::FLOAT[{model.get_sentence_embedding_dimension()}]) as distance 
        FROM {table_name}
        ORDER BY distance 
        LIMIT {k};
    """
    ).to_df()


df = similarity_search_with_duckdb_index(query="I wanna know what love is")
for row in df.iterrows():
    prompt_wrapped = "\n".join(textwrap.wrap(row[1]["prompt"], width=100))
    response_wrapped = "\n".join(textwrap.wrap(row[1]["response"], width=100))
    print(f'😎 {prompt_wrapped}')
    print(f'\n🤖 {response_wrapped}')
    print("")

😎 The paragraph below is from an alternative version of the same song, which was written by another
rapper, Chamillionaire. Inspired by the lyrics, continue the essay you wrote before, connecting the
topic of party and hedonism with love and romantic relationships. In this second part of the essay
(no longer than 1200 characters), mention serious philosophical references as well as the pop
culture ones in the lyrics (rock vs pop music, DJs and video influencers, eCommerce websites such as
eBay...), and try to answer the following question: "Is romantic relationship just a form of
Hedonism?"  I’m a rock star, not a pop star Pay the toll and hop on my freeway I’m up in the club
tryna’ purchase love Like I couldn’t find it on eBay And she feelin' freaky, light some candles Like
it's her bday  She wanna hear Weezer on replay For Pete's sake what I look like I’m a DJ Wanna be on
TV, what you think I got my camera for? Grab the patron and I stand on the bar Cause I already
sipped on a gallon

In [17]:
df = similarity_search_with_duckdb_index(query="should chatbots apologize?", k=1000)
for row in df.head(5).iterrows():
    prompt_wrapped = "\n".join(textwrap.wrap(row[1]["prompt"], width=100))
    response_wrapped = "\n".join(textwrap.wrap(row[1]["response"], width=100))
    print(f'😎 {prompt_wrapped}')
    print(f'\n🤖 {response_wrapped}')
    print("")

😎 Should chatbots apologize?

🤖 Chatbots are designed to follow specific guidelines set by their developers, including how they
interact with users. The question of whether chatbots should apologize hinges on the purpose of the
apology and the context in which it is given. Here are a few considerations:  1. **User
Experience**: An apology from a chatbot can be part of creating a polite and user-friendly
interaction. It can make the chatbot seem more relatable and improve the overall user experience by
acknowledging errors or misunderstandings in a conversation.  2. **Clarification and Trust**: When a
chatbot misunderstands a question or provides incorrect information, acknowledging the mistake can
clarify the situation for the user and help maintain trust in the chatbot's ability to provide
useful information.  3. **Programming and Limitations**: Chatbots are programmed to respond based on
the data and rules provided by their developers. They do not possess emotions or self-awareness, 

In [18]:
df = similarity_search_with_duckdb_index(query="who is reddgr?", k=1000)
for row in df.head(5).iterrows():
    prompt_wrapped = "\n".join(textwrap.wrap(row[1]["prompt"], width=100))
    response_wrapped = "\n".join(textwrap.wrap(row[1]["response"], width=100))
    print(f'😎 {prompt_wrapped}')
    print(f'\n🤖 {response_wrapped}')
    print("")

😎 Reddgr

🤖 summaryReddgrturn0search0 Reddgr is a personal knowledge-sharing project created by David González
Romero, an ICT engineer and business consultant with over 15 years of experience in enterprise
software solutions, consulting, business analytics, and data science. citeturn0search4 The project
encompasses a blog and the "Talking to Chatbots" platform, focusing on the intersection of
technology, culture, and business, with a particular emphasis on artificial intelligence (AI) and
generative AI. citeturn0search1  The blog features diverse content, including discussions on AI
ethics, financial insights, and analyses of song meanings. It aims to provide in-depth, lasting
content that contrasts with the fleeting nature of social media updates. The "Talking to Chatbots"
section showcases conversations with AI chatbots, exploring topics such as science fiction, music,
and professional development. citeturn0search0  Reddgr maintains an active presence on various
social media platform

In [19]:
df = similarity_search_with_duckdb_index(query="what is the meaning of life?", k=1000)
for row in df.head(3).iterrows():
    prompt_wrapped = "\n".join(textwrap.wrap(row[1]["prompt"], width=100))
    response_wrapped = "\n".join(textwrap.wrap(row[1]["response"], width=100))
    print(f'😎 {prompt_wrapped}')
    print(f'\n🤖 {response_wrapped}')
    print("")

😎 Briefly describe what a person being “empty” means

🤖 Being "empty" typically refers to a state where a person feels a lack of emotions, meaning, or
purpose, often associated with feelings of numbness or disconnection from oneself and others.

😎 Explain what it can mean

🤖 The phrase "The more you claim expertise, the more your need for money shows" suggests that a
person's frequent assertions of their own expertise could be perceived as an indirect indication of
their primary motivation being financial gain, rather than genuine skill or knowledge. It implies a
skepticism towards the authenticity of the expertise claimed by individuals who vocally and
repeatedly assert their proficiency or knowledge in a particular area, hinting that such assertions
might be more about soliciting financial opportunities than about the expertise itself.

😎 “Reality is that which, when you stop believing in it, doesn’t go away”  Attribute the quote and
explain its meaning

🤖 This quote is attributed to